In [1]:
from pathlib import Path
import pandas as pd
import geopandas as gpd

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 240)

here = Path.cwd()
ROOT = here if (here / "data" / "interim").exists() else here.parent
INTERIM = ROOT / "data" / "interim"
PROCESSED = ROOT / "data" / "processed"
PROCESSED.mkdir(parents=True, exist_ok=True)

# list the cleaned files we are about to combine
files = sorted(INTERIM.glob("clean_*.gpkg"))
print("found", len(files), "cleaned files:")
for f in files:
    print("  ", f.name)

found 4 cleaned files:
   clean_2022.gpkg
   clean_2023.gpkg
   clean_2024.gpkg
   clean_2025.gpkg


In [2]:
# Read each cleaned file and check they all match before combining.
frames = []
col_sets = []
for f in files:
    g = gpd.read_file(f)
    frames.append(g)
    col_sets.append(tuple(g.columns))
    print(f"{f.name}: {len(g)} rows | CRS {g.crs} | {len(g.columns)} cols")

# safety check: do all files have the exact same columns?
print()
if len(set(col_sets)) == 1:
    print("COLUMN CHECK: all 4 files have identical columns. Safe to combine.")
else:
    print("COLUMN CHECK: MISMATCH. Columns differ between files:")
    for f, cs in zip(files, col_sets):
        print(" ", f.name, "->", cs)

clean_2022.gpkg: 55 rows | CRS EPSG:4326 | 23 cols
clean_2023.gpkg: 60 rows | CRS EPSG:4326 | 23 cols
clean_2024.gpkg: 323 rows | CRS EPSG:4326 | 23 cols
clean_2025.gpkg: 769 rows | CRS EPSG:4326 | 23 cols

COLUMN CHECK: all 4 files have identical columns. Safe to combine.


In [3]:
# Stack all 4 years into one master table.
master = pd.concat(frames, ignore_index=True)
master = gpd.GeoDataFrame(master, geometry="geometry", crs=frames[0].crs)

print("master rows:", len(master), "(expected 1207)")
print("master columns:", len(master.columns))
print()

# rows per year (sanity check)
print("rows per year:")
print(master["year"].value_counts().sort_index())
print()

# check no field ID is duplicated across the whole dataset
dups = master["uid"].duplicated().sum()
print("duplicate UIDs:", dups, "(expected 0)")

master rows: 1207 (expected 1207)
master columns: 23

rows per year:
year
2022     55
2023     60
2024    323
2025    769
Name: count, dtype: int64

duplicate UIDs: 0 (expected 0)


In [4]:
# Save the combined master table. This is the single file all future steps read from.
out = PROCESSED / "yield_master.gpkg"
master.to_file(out, driver="GPKG")

print("saved:", out)
print("total rows:", len(master))
print("columns:", len(master.columns))
print()
print("rows per year:")
print(master["year"].value_counts().sort_index())
print()
print("flagged fields (qc_flag not empty):", (master["qc_flag"] != "").sum())

saved: f:\THESIS\mtech_thesis_crop_yield_estimation\data\processed\yield_master.gpkg
total rows: 1207
columns: 23

rows per year:
year
2022     55
2023     60
2024    323
2025    769
Name: count, dtype: int64

flagged fields (qc_flag not empty): 36
